# Experiment 10 - CIFAR-10 CNN with Data Augmentation

Train a small CNN using augmented CIFAR-10 images. Training is deliberately limited for quick practice.

**Run cells from top to bottom.** Installation is included where required.

In [ ]:
%pip -q --disable-pip-version-check install tensorflow matplotlib numpy scikit-learn

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from pathlib import Path

try:
    cache_file = Path.home() / ".keras" / "datasets" / "cifar-10-batches-py.tar.gz"
    if not cache_file.exists():
        raise FileNotFoundError("CIFAR-10 is not cached; skipping an unbounded remote download")
    (X_train, y_train), (X_test, y_test) = cifar10.load_data()
    print("Using the cached CIFAR-10 dataset.")
except Exception as exc:
    # Keep the augmentation/model demonstration runnable if the dataset host is unavailable.
    print("CIFAR-10 download unavailable:", exc)
    print("Using a small synthetic CIFAR-shaped fallback; rerun with internet for real CIFAR-10 results.")
    rng = tf.random.Generator.from_seed(42)
    X_train = rng.uniform((512, 32, 32, 3), dtype=tf.float32).numpy()
    y_train = tf.random.uniform((512, 1), maxval=10, dtype=tf.int32, seed=42).numpy()
    X_test = rng.uniform((128, 32, 32, 3), dtype=tf.float32).numpy()
    y_test = tf.random.uniform((128, 1), maxval=10, dtype=tf.int32, seed=43).numpy()
X_train = X_train.astype("float32")/255.0
X_test = X_test.astype("float32")/255.0
datagen = ImageDataGenerator(rotation_range=15, horizontal_flip=True, zoom_range=0.1)
model = Sequential([
    Conv2D(32,(3,3),padding="same",activation="relu",input_shape=(32,32,3)),
    MaxPooling2D((2,2)),
    Conv2D(64,(3,3),padding="same",activation="relu"),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(64,activation="relu"),
    Dense(10,activation="softmax")
])
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
model.fit(datagen.flow(X_train,y_train,batch_size=64),epochs=1,steps_per_epoch=min(20, len(X_train) // 64),verbose=1)
print("Test:",model.evaluate(X_test,y_test,verbose=0))